
# Simple ISIN → `yfinance.Ticker(ISIN).ticker`

This notebook does exactly what was requested:
- Reads your CSV.
- For rows where **Stock ID** is non-empty, it runs:
  ```python
  import yfinance as yf
  yf.Ticker(ISIN).ticker
  ```
- Writes the result to a new column **`Yahoo ticker`**.
- Prints how many rows "found" a ticker, how many did not, and the percentage.

> **Heads-up:** `yfinance.Ticker("ISIN").ticker` **does not resolve** an ISIN to a Yahoo symbol;
> it returns the exact string you pass in. So if `Stock ID` contains an ISIN, the output will
> generally be the same ISIN. This notebook follows the requested behavior verbatim.


In [5]:
# --- config ---
IN_CSV   = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\pre_data\uni_pre_cleaned.csv"
OUT_CSV  = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\pre_data\uni_with_yf_tickers_MIN_with_LC.csv"
LOOKUP_CSV = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\yahoo_suffix_mapping_full.csv"

ISIN_COL = "Stock ID"
NAME_COL = "Name/Kind of Investment Item"   # change to "Name" if that's your header

MAX_WORKERS = 16  # 12–20 is usually fine

# --- imports ---
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    import yfinance as yf
except ImportError:
    raise SystemExit("Please: pip install yfinance")

# ------------------------------- helpers --------------------------------
def get_ticker_from_isin(isin: str) -> tuple[str, str]:
    """
    Returns (isin, resolved_ticker_or_empty).
    Touch .fast_info once to nudge yfinance to hit Yahoo, then read .ticker.
    """
    try:
        t = yf.Ticker(isin)
        try:
            _ = t.fast_info  # may raise; that's fine
        except Exception:
            pass
        sym = (t.ticker or "").strip()
        return isin, sym
    except Exception:
        return isin, ""

def read_lookup_frame(path: str) -> pd.DataFrame:
    # try common encodings
    for enc in ("utf-8-sig", "cp1252"):
        try:
            return pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[], encoding=enc)
        except Exception:
            pass
    raise RuntimeError(f"Could not read lookup CSV: {path}")

def build_suffix_map(lookup_df: pd.DataFrame) -> tuple[dict, str]:
    """
    Returns:
      - suffix_map: dict 'SUFFIX' -> 'Country' (suffix normalized, no leading dot)
      - default_lc: the Country from the FIRST ROW of the lookup file (for tickers with no dot)
    """
    if lookup_df.empty:
        return {}, "US"

    cols = {c.lower().strip(): c for c in lookup_df.columns}
    suffix_col  = cols.get("suffix")   or list(cols.values())[0]
    country_col = cols.get("country")  or list(cols.values())[2]

    # default LC for no-suffix tickers: Country value from the FIRST ROW (as you requested)
    default_lc = str(lookup_df.iloc[0][country_col]).strip()

    lk = lookup_df.copy()
    lk["_SUF"] = (
        lk[suffix_col].astype(str)
                      .str.strip()
                      .str.upper()
                      .str.lstrip(".")   # ".TO" -> "TO"
    )

    suffix_map = (
        lk[["_SUF", country_col]]
          .drop_duplicates(subset=["_SUF"])
          .set_index("_SUF")[country_col]
          .to_dict()
    )
    # If the first row's suffix was blank, ensure "" maps to that first row's country
    if lk.iloc[0]["_SUF"] == "":
        suffix_map[""] = default_lc

    return suffix_map, default_lc

def extract_suffix(symbol: str) -> str:
    if not isinstance(symbol, str): return ""
    s = symbol.strip()
    if "." not in s: return ""
    return s.rsplit(".", 1)[-1].upper()

def lc_for_symbol(symbol: str, suffix_map: dict, default_lc: str) -> str:
    if not isinstance(symbol, str) or not symbol.strip():
        return ""
    suf = extract_suffix(symbol)
    if suf == "":
        # no dot → treat as the first row in your suffix table
        return default_lc
    return suffix_map.get(suf, "")  # blank if unknown suffix

# --------------------------------- main ----------------------------------
def main():
    # 1) load
    df = pd.read_csv(IN_CSV, dtype=str, keep_default_na=False, na_values=[], encoding="cp1252")

    mask = df.get(ISIN_COL, pd.Series([""]*len(df))).astype(str).str.strip().ne("")
    isins_series = df.loc[mask, ISIN_COL].astype(str).str.strip()

    # de-dup to reduce calls
    unique_isins = list(dict.fromkeys(isins_series))

    # 2) parallel fetch yfinance
    mapping = {}
    if unique_isins:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futs = [ex.submit(get_ticker_from_isin, i) for i in unique_isins]
            for n, fut in enumerate(as_completed(futs), 1):
                k, v = fut.result()
                mapping[k] = v
                if n % 200 == 0 or n == len(unique_isins):
                    print(f"{n}/{len(unique_isins)} ISINs processed")

    # 3) write Yahoo ticker column
    df["Yahoo ticker"] = ""
    df.loc[mask, "Yahoo ticker"] = isins_series.map(lambda s: mapping.get(s, ""))

    # 4) slim output (only rows we processed): Name, ISIN, Yahoo ticker
    df_out = df.loc[mask, [NAME_COL, ISIN_COL, "Yahoo ticker"]].copy()
    df_out = df_out.rename(columns={NAME_COL: "Name"})  # optional rename

    # 5) add LC using your suffix lookup
    lk = read_lookup_frame(LOOKUP_CSV)
    suffix_map, default_lc = build_suffix_map(lk)
    df_out["LC"] = df_out["Yahoo ticker"].apply(lambda sym: lc_for_symbol(sym, suffix_map, default_lc))

    # quick report on suffixes we couldn't map (useful to improve the table)
    unmapped = (
        df_out.loc[df_out["Yahoo ticker"].astype(str).str.contains(r"\.", regex=True) & (df_out["LC"] == "")]
              ["Yahoo ticker"].map(extract_suffix).value_counts()
    )
    if not unmapped.empty:
        print("Unmapped Yahoo suffixes (top 20):")
        print(unmapped.head(20).to_string())

    # 6) save + summary
    df_out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    tickers = df_out["Yahoo ticker"].astype(str).str.strip()
    isins   = df_out[ISIN_COL].astype(str).str.strip()

    found_any = (tickers != "").sum()
    resolved  = ((tickers != "") & (tickers.str.upper() != isins.str.upper())).sum()
    echoed    = ((tickers != "") & (tickers.str.upper() == isins.str.upper())).sum()
    eligible  = len(df_out)
    not_found = eligible - found_any

    print("✅ Saved:", OUT_CSV)
    print("---- Summary ----")
    print(f"Rows with ISIN (processed): {eligible}")
    print(f"Returned any text:          {found_any}")
    print(f"Resolved to a real ticker:  {resolved}  (ticker != ISIN)")
    print(f"Echoed ISIN back:           {echoed}    (ticker == ISIN)")
    print(f"No ticker:                  {not_found}")
    if eligible:
        print(f"Hit rate (any):             {found_any/eligible*100:.2f}%")
        print(f"Hit rate (resolved):        {resolved/eligible*100:.2f}%")

if __name__ == "__main__":
    main()


200/3234 ISINs processed
400/3234 ISINs processed
600/3234 ISINs processed
800/3234 ISINs processed
1000/3234 ISINs processed
1200/3234 ISINs processed
1400/3234 ISINs processed
1600/3234 ISINs processed
1800/3234 ISINs processed
2000/3234 ISINs processed
2200/3234 ISINs processed
2400/3234 ISINs processed
2600/3234 ISINs processed
2800/3234 ISINs processed
3000/3234 ISINs processed
3200/3234 ISINs processed
3234/3234 ISINs processed
Unmapped Yahoo suffixes (top 20):
Yahoo ticker
AE    3
BD    1
✅ Saved: D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\pre_data\uni_with_yf_tickers_MIN_with_LC.csv
---- Summary ----
Rows with ISIN (processed): 3252
Returned any text:          3188
Resolved to a real ticker:  3188  (ticker != ISIN)
Echoed ISIN back:           0    (ticker == ISIN)
No ticker:                  64
Hit rate (any):             98.03%
Hit rate (resolved):        98.03%
